# LACE-VQA: lightweight VQA for blind and low-vision users — full experiment run

Runtime: **GPU (T4)**. Every stage is resumable: if Colab disconnects, re-run the *Setup* cells and then the stage you were on.
Results are mirrored to Google Drive after every stage.

Rough time on a free T4 (not measured on this exact setup — treat as estimates): download 30–60 min, features 20–40 min,
main training 15–30 min, ablations + backbones 2–3 h, zero-shot + robustness 2–4 h, benchmark 20 min.

## 1. Setup (run every session)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/access-vqa'   # code zip + backups live here
import os; os.makedirs(DRIVE, exist_ok=True)

In [ ]:
# Upload access-vqa.zip to MyDrive/access-vqa/ once, then:
!cd /content && rm -rf access-vqa && unzip -q -o $DRIVE/access-vqa.zip -d /content && ls /content/access-vqa
%cd /content/access-vqa
!pip install -q -r requirements.txt
import torch; print(torch.__version__, torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU')

In [ ]:
# Work directory on the fast local disk; restore the backup from Drive if one exists.
%env DATA=/content/data
%env WORK=/content/work
%env WORKERS=2
%env ZIP_CACHE=/content/drive/MyDrive/access-vqa/zips
!mkdir -p /content/work && rsync -a $DRIVE/work/ /content/work/ 2>/dev/null; ls /content/work
def backup():
    !rsync -a /content/work/ $DRIVE/work/

## 2. Optional: 5-minute smoke test (tiny synthetic data, checks the code runs)

In [ ]:
!bash tests/smoke_test.sh 2>&1 | tail -3

## 3. Get VizWiz-VQA
First session: downloads once and keeps the zips in Drive (`ZIP_CACHE`). Later sessions copy from Drive (minutes, no re-download). After features exist, sessions that only need validation images can run `scripts/download_data.sh vizwiz_val` instead.

In [ ]:
!bash scripts/download_data.sh vizwiz
!ls /content/data/vizwiz /content/data/vizwiz/Annotations

## 4. Extract CLIP ViT-B/16 features, then train the main models (3 seeds)

In [ ]:
!STAGES="feat_viz" bash scripts/run_all.sh
backup()

In [ ]:
!STAGES="train_main" bash scripts/run_all.sh 2>&1 | grep -E "^\{|TEST|train |trainable|skip"
backup()

## 5. Ablations and the encoder-size study

In [ ]:
!STAGES="ablations" bash scripts/run_all.sh 2>&1 | grep -E "TEST|skip|accuracy\"" | tail -40
backup()

In [ ]:
!STAGES="backbones" bash scripts/run_all.sh 2>&1 | grep -E "TEST|skip|wrote" | tail -20
backup()

## 6. Zero-shot baselines, robustness, INT8 accuracy

In [ ]:
!STAGES="zeroshot" bash scripts/run_all.sh 2>&1 | grep -E '^\{"model' 
backup()

In [ ]:
!STAGES="robustness" bash scripts/run_all.sh 2>&1 | grep -E '^\{"model'
backup()

In [ ]:
!STAGES="int8acc" bash scripts/run_all.sh 2>&1 | grep -E '^\{"model'
backup()

## 7. Efficiency benchmark (run on a fresh runtime with nothing else running, for clean timings)

In [ ]:
!STAGES="benchmark" bash scripts/run_all.sh
backup()

## 8. Optional: VQA v2 pre-training (zips cached in Drive too; needs ~40 GB of local Colab disk while extracting)

In [ ]:
!bash scripts/download_data.sh vqav2
!STAGES="feat_vqa vqa_pretrain" bash scripts/run_all.sh 2>&1 | grep -E "TEST|skip|wrote"
backup()

## 9. Fill the paper tables and figures, and download everything

In [ ]:
!STAGES="tables" bash scripts/run_all.sh
!cd paper && latexmk -pdf -interaction=nonstopmode main.tex > /dev/null 2>&1; ls paper
!cd /content && zip -qr results_bundle.zip access-vqa/paper work/runs/*/results.json work/runs/*/log.jsonl work/results
!cp /content/results_bundle.zip $DRIVE/
print('Saved to Drive:', DRIVE + '/results_bundle.zip')